In [6]:
import pickle
import os
# load env
from dotenv import load_dotenv
load_dotenv()

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA


In [3]:
with open("data.pkl", "rb") as f:
    data = pickle.load(f)

In [4]:
data

{1: "ey RE « sutrorzation/ Prior Authorization Request Form : ~ vp te “Complete all Sections to ensure timely review ---...: ao : .. aU t service(s) are related to Cancer, Specialty Medications or Behavioral Health please use ‘the designated form 4 *All preauthorization requests must be submitted with supporting clinical documentation that is relevant to the request. Forms will be returned if not fille ingly or if they are submitted withaut the required clinical information. Fax to r for Urgent Services fax t Provider appeals submitted on this form will not be considered, Please use the claim resubmission request form found on or our ir website. - Section A: Request information ee he ee ST DE a DT is Today's Date: 98/29/2022 Completed _ | Decisions on preauthorization requests submitted with all necessary clinical information will be made within 15 calendar days of receipt of the request. Bi Service is Scheduled (only if applicable) Schedule Date: 10/21/2022 DO Urgent Request (only if 

In [ ]:


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)
all_documents = []
for k, v in data.items():
    print(f"Processing page {k}")
    chunks = text_splitter.split_text(v)
    for chunk in chunks:
        all_documents.append(Document(page_content=chunk, metadata={"page": k, "source": "input_pdf"}))

Processing page 1
Processing page 2
Processing page 3
Processing page 4
Processing page 5


In [57]:
all_documents

[Document(metadata={'page': 1, 'source': 'input_pdf'}, page_content='ey RE « sutrorzation/ Prior Authorization Request Form : ~ vp te “Complete all Sections to ensure timely review ---...: ao : .. aU t service(s) are related to Cancer, Specialty Medications or Behavioral Health please use ‘the designated form 4 *All preauthorization requests must be submitted with supporting clinical documentation that is relevant to the request. Forms will be returned if not fille ingly or if they are submitted withaut the required clinical information. Fax to r for Urgent'),
 Document(metadata={'page': 1, 'source': 'input_pdf'}, page_content="fille ingly or if they are submitted withaut the required clinical information. Fax to r for Urgent Services fax t Provider appeals submitted on this form will not be considered, Please use the claim resubmission request form found on or our ir website. - Section A: Request information ee he ee ST DE a DT is Today's Date: 98/29/2022 Completed _ | Decisions on pr

In [ ]:
embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# vectorstore = FAISS.from_documents(all_documents, embeddings)

# vectorstore.save_local("faiss_index")

vectorstore = FAISS.load_local("faiss_index", embeddings)

In [ ]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    max_tokens=1000,
    api_key=os.getenv("OPENAI_API_KEY"),
    request_timeout=60
)

In [ ]:
query = "What is the main topic of the document?"
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(query)

In [ ]:
docs

[Document(id='ca327650-8b8c-487f-831e-f9d85b635f21', metadata={'page': 5}, page_content='medical record and documentation was completed utilizing voice recognition software. As a result and despite that editing was performed, there may exist inadvertent grammatical and spelling errors associated to deficits in the dictation/voice- recognition software. Electronically signed ~ fe on 09/08/2022 at 7:00 AM'),
 Document(id='2d89914f-a438-4a56-94ff-4ba158624cf0', metadata={'page': 2}, page_content='MD Primary Provider: MD, Chief Complaint: Lt hand Dominate Hand: Right Hand History of Present Iliness: The patient returns to the office today for concern of pin migration. The patient has now receded to below skin level. It is not productive of much pain. There are no signs of infection currently. Review of Systems MS Complains of swelling of joints, Physical Exam Musculoskeletal: On focused physical examination the patient soft tissues are healing appropriately without signs of infection.'),
 

In [ ]:

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [51]:
result = qa.invoke("What is the CPT code for this case?")

In [52]:
result["result"]

'The CPT code for the right carpal tunnel release and right carpometacarpal arthroplasty mentioned in the context is 64721 and 25447, respectively.'

In [ ]:
source_docs = result["source_documents"]
source_docs_metadata = [doc.metadata for doc in source_docs]

In [ ]:
source_docs_metadata

[{'page': 5}, {'page': 2}, {'page': 1}]

In [ ]:
# %%writefile ../modules/rag.py


from langchain.chains import RetrievalQA
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
load_dotenv()


class RagPipeline:
    def __init__(self, doc_data, doc_name="input_pdf"):
        self.doc_data = doc_data
        self.doc_name = doc_name
        self.vectorstore = None

    # def _check_if_index_exists(self):
    #     if os.path.exists("faiss_index"):
    #         print("Index already exists. Loading from disk.")
    #         self.vectorstore = FAISS.load_local("faiss_index", OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY")))
    #     else:
    #         print("Index does not exist. Building index.")
    #         self.build_index()

    def build_index(self):
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
            length_function=len
        )
        all_documents = []
        for k, v in self.doc_data.items():
            chunks = text_splitter.split_text(v)
            for chunk in chunks:
                all_documents.append(Document(page_content=chunk, metadata={
                                     "page": k, "source": self.doc_name}))

        embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))
        self.vectorstore = FAISS.from_documents(all_documents, embeddings)
        self.vectorstore.save_local(
            "vector_data/" + self.doc_name + "_faiss_index")


class QueryRAG:
    def __init__(self, doc_name="input_pdf"):
        self.doc_name = doc_name
        self.vectorstore = FAISS.load_local("vector_data/" + self.doc_name + "_faiss_index", OpenAIEmbeddings(
            api_key=os.getenv("OPENAI_API_KEY")), allow_dangerous_deserialization=True)
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
        self.llm = ChatOpenAI(
            model="gpt-3.5-turbo",
            temperature=0,
            max_tokens=1000,
            api_key=os.getenv("OPENAI_API_KEY"),
            request_timeout=60
        )
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            retriever=self.retriever,
            return_source_documents=True
        )

    def query(self, question):
        result = self.qa_chain.invoke(question)
        unique_documents = {
            doc.metadata["source"]: doc for doc in result["source_documents"]}.values()
        return result["result"], unique_documents


Overwriting ../modules/rag.py


In [14]:
rag_pipeline = RagPipeline(data)
rag_pipeline.build_index()

In [11]:
query_rag = QueryRAG()
result, source_docs = query_rag.query("What is the CPT code for this case?")

In [12]:
result

'The CPT code for the right carpal tunnel release and right carpometacarpal arthroplasty mentioned in the context is 64721 and 25447, respectively.'